# Chapter 2 lab — Measuring the gait stability/speed trade-off

Companion to **[Chapter 2 — Legs and Fingers: Nature's Engineering](https://kamatechorg.github.io/robo-greeno-data-a/tutorial/02-legs-and-fingers/)**.

Chapter 2 says our simulation implements the tripod, wave, and ripple gaits *“so you can measure the stability-speed trade-off instead of taking nature's word for it.”* This is that measurement.

Pure Python + matplotlib — **no GPU, no MuJoCo, ~2 minutes.** We model each gait as a set of leg phase offsets, then count how many feet are on the ground at every instant (static stability needs ≥ 3 in a supporting triangle).

## Step 1 — define the three gaits

A gait is just a **duty factor** (fraction of the cycle each foot is on the ground) plus a **phase offset** per leg (when in the cycle it lifts). Legs are numbered:

```
    1 ---- torso ---- 2     (front)
    3 ---- torso ---- 4     (middle)
    5 ---- torso ---- 6     (back)
```

In [ ]:
import numpy as np

# Tripod: two sets of 3 legs, 180 deg out of phase, each foot down half the time.
# Wave: one leg at a time, each foot down 5/6 of the time (slowest, most stable).
# Ripple: overlapping pairs, a compromise.
GAITS = {
    "tripod": dict(duty=0.5,  phases={1:0.0, 4:0.0, 5:0.0,  2:0.5, 3:0.5, 6:0.5}),
    "wave":   dict(duty=5/6,  phases={1:0.0, 3:1/6, 5:2/6, 2:3/6, 4:4/6, 6:5/6}),
    "ripple": dict(duty=2/3,  phases={1:0.0, 6:1/6, 3:2/6, 4:3/6, 5:4/6, 2:5/6}),
}

def feet_down(gait, t):
    """Return the set of legs whose foot is on the ground at cycle-phase t in [0,1)."""
    duty, phases = gait["duty"], gait["phases"]
    down = set()
    for leg, off in phases.items():
        # foot is down during the first `duty` fraction of its shifted cycle
        if ((t - off) % 1.0) < duty:
            down.add(leg)
    return down

print("sanity check at t=0:")
for name, g in GAITS.items():
    print(f"  {name:7s} feet down: {sorted(feet_down(g, 0.0))}")

## Step 2 — count feet on the ground across a full cycle

Static stability requires at least **3 non-collinear feet** down. The *minimum* number of feet down over the cycle is the gait's stability margin: the higher, the safer.

In [ ]:
T = np.linspace(0, 1, 200, endpoint=False)
summary = {}
for name, g in GAITS.items():
    counts = np.array([len(feet_down(g, t)) for t in T])
    summary[name] = counts
    print(f"{name:7s}  feet down: min={counts.min()}  mean={counts.mean():.2f}  "
          f"-> {'STATICALLY STABLE' if counts.min() >= 3 else 'NEEDS DYNAMIC BALANCE'}")

## Step 3 — plot the support patterns

Black = foot on the ground (supporting), white = swing (in the air). Read each row as one leg over one full stride.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
legs = [1, 2, 3, 4, 5, 6]
for ax, (name, g) in zip(axes, GAITS.items()):
    grid = np.array([[1 if leg in feet_down(g, t) else 0 for t in T] for leg in legs])
    ax.imshow(grid, aspect="auto", cmap="gray_r", extent=[0, 1, 6.5, 0.5])
    ax.set_title(f"{name}  (min {summary[name].min()} feet down)")
    ax.set_yticks(legs); ax.set_xlabel("stride phase")
axes[0].set_ylabel("leg #")
plt.tight_layout(); plt.show()

## Step 4 — the trade-off, quantified

Speed proxy: how much of the cycle a leg spends *swinging* (propelling) — i.e. `1 - duty`. More swing time = faster potential, fewer feet down = less stable.

In [ ]:
print(f"{'gait':8s}{'min feet down':>14s}{'swing fraction':>16s}{'character':>22s}")
notes = {"tripod":"fast, just-stable", "wave":"slowest, safest", "ripple":"balanced"}
for name, g in GAITS.items():
    swing = 1 - g["duty"]
    print(f"{name:8s}{summary[name].min():>14d}{swing:>16.2f}{notes[name]:>22s}")

!!! abstract "What you just measured"
    - **Wave** keeps 5 feet down — maximum stability, minimum swing (slow).
    - **Tripod** sits right at the edge: exactly 3 feet down at all times — the most swing time (fastest) while *still never needing to balance*. That edge is why Chapter 2 calls six legs “the best first robot.”
    - **Ripple** lands in between, as advertised.

**Try it:** invent your own gait by editing the `phases` dict, and check whether `min feet down` stays ≥ 3. Next: **[Chapter 3 — MuJoCo Without Tears](https://kamatechorg.github.io/robo-greeno-data-a/tutorial/03-mujoco-intuition/)**.